# PPP 2021 Global 1000-Bins Replication (Documented)

This notebook reproduces the Stata pipeline as far as possible with the files currently available in this workspace.

## Goal
- Replicate the global 1000-bins construction workflow under PPP 2021.
- Make each step transparent and fully documented.

## Important limitation
The Stata scripts require intermediate files such as `FullDistributions.dta` and `CollapsedDistributions.dta`. Those files are not present in this workspace, so we provide a **partial but auditable replication** using available PPP 2021 files.

## Chunk 1 - Setup and imports
This chunk imports packages, sets display options, and defines project paths.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

ROOT = Path('.').resolve()
INPUT = ROOT / '01-input'
ROOT, INPUT

(WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/01-input'))

## Chunk 2 - Locate required PPP 2021 files
This chunk finds the PPP 2021 `fillgaps` and `GlobalDist1000bins` files and checks whether intermediate Stata outputs exist.

In [2]:
fillgaps_candidates = [
    INPUT / '20260922' / 'lineup' / 'fillgaps.dta',
    INPUT / 'nuevosyuki' / 'fillgaps.dta',
]
globaldist_candidates = [
    INPUT / '20260922' / 'lineup' / 'GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta',
    INPUT / 'nuevosyuki' / 'GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta',
]

fillgaps_path = next((p for p in fillgaps_candidates if p.exists()), None)
globaldist_path = next((p for p in globaldist_candidates if p.exists()), None)
interpolated_path = INPUT / 'interpolated_means.dta'

fulldist_path = next((p for p in ROOT.rglob('*FullDistributions*.dta')), None)
collapsed_path = next((p for p in ROOT.rglob('*CollapsedDistributions*.dta')), None)

pd.DataFrame({
    'item': ['fillgaps', 'globaldist', 'interpolated_means', 'FullDistributions', 'CollapsedDistributions'],
    'path': [fillgaps_path, globaldist_path, interpolated_path, fulldist_path, collapsed_path],
    'exists': [fillgaps_path is not None, globaldist_path is not None, interpolated_path.exists(), fulldist_path is not None, collapsed_path is not None]
})

,item,path,exists
0,fillgaps,C:\Users\wb661551\OneDrive - WBG\Desktop\Inter...,True
1,globaldist,C:\Users\wb661551\OneDrive - WBG\Desktop\Inter...,True
2,interpolated_means,C:\Users\wb661551\OneDrive - WBG\Desktop\Inter...,True
3,FullDistributions,None,False
4,CollapsedDistributions,None,False


## Chunk 3 - Load available inputs
This chunk loads available datasets and harmonizes data types for year fields.

In [3]:
if fillgaps_path is None or globaldist_path is None:
    raise FileNotFoundError('PPP 2021 fillgaps/globaldist files not found.')

fillgaps = pd.read_stata(fillgaps_path, convert_categoricals=False)
globaldist = pd.read_stata(globaldist_path, convert_categoricals=False)
interpolated = pd.read_stata(interpolated_path, convert_categoricals=False)

for df in (fillgaps, globaldist, interpolated):
    if 'year' in df.columns:
        df['year'] = df['year'].astype(int)

print('fillgaps shape:', fillgaps.shape)
print('globaldist shape:', globaldist.shape)
print('interpolated shape:', interpolated.shape)

fillgaps shape: (10120, 24)
globaldist shape: (8066000, 8)
interpolated shape: (10281, 42)


## Chunk 4 - PPP 2021 consistency diagnostics
This chunk checks whether PPP-related metadata aligns across available files.

In [4]:
def ppp_like_cols(df):
    keys = ['ppp', 'cpi', 'icp', 'price', 'usd', 'lcu']
    return [c for c in df.columns if any(k in c.lower() for k in keys)]

ppp_report = {
    'fillgaps_ppp_cols': ppp_like_cols(fillgaps),
    'globaldist_ppp_cols': ppp_like_cols(globaldist),
    'interpolated_ppp_cols': ppp_like_cols(interpolated),
    'globaldist_pipvintage_values': sorted(globaldist['pipvintage'].dropna().astype(str).unique().tolist())[:10]
}
ppp_report

{'fillgaps_ppp_cols': [],
 'globaldist_ppp_cols': [],
 'interpolated_ppp_cols': ['predicted_mean_ppp',
  'survey_mean_lcu',
  'survey_mean_ppp',
  'ppp',
  'cpi',
  'cpi_data_level',
  'ppp_data_level'],
 'globaldist_pipvintage_values': ['20260922_2021_01_02_PROD']}

## Chunk 5 - Build `pop_all` analog (from fillgaps)
Stata script builds population and region lookup. This chunk recreates that lookup from `fillgaps` using national rows (plus ARG/SUR logic).

In [5]:
pop_all = fillgaps[[
    'country_code', 'region_code', 'reporting_level', 'year', 'population'
]].copy()

pop_all = pop_all[
    (pop_all['reporting_level'] == 'national')
    | (pop_all['country_code'].isin(['ARG', 'SUR']))
]
pop_all = pop_all[(pop_all['year'] > 1980) & (pop_all['year'] < 2020)]
pop_all = pop_all.dropna(subset=['country_code', 'year', 'population'])

# Stata conversion: country pop to millions.
pop_all['pop_millions'] = pop_all['population'] / 1_000_000
pop_all = pop_all.drop_duplicates(['country_code', 'reporting_level', 'year'])

pop_all.head(10)

,country_code,region_code,reporting_level,year,population,pop_millions
0,ABW,LCN,national,1981,60563,0.060563
1,ABW,LCN,national,1982,61276,0.061276
2,ABW,LCN,national,1983,62228,0.062228
3,ABW,LCN,national,1984,62901,0.062901
4,ABW,LCN,national,1985,61728,0.061728
5,ABW,LCN,national,1986,59931,0.059931
6,ABW,LCN,national,1987,59159,0.059159
7,ABW,LCN,national,1988,59331,0.059331
8,ABW,LCN,national,1989,60443,0.060443
9,ABW,LCN,national,1990,62753,0.062753


## Chunk 6 - Approximate final GlobalDist output frame
This chunk maps `quantile` to bin number (`obs`) and attaches region information. It is equivalent to the final shape expected from global 1000-bins outputs.

In [8]:
gd = globaldist.copy()
gd = gd.rename(columns={'quantile': 'obs'})
gd['obs'] = gd['obs'].astype(int)

region_lookup = pop_all[['country_code', 'year', 'region_code']].drop_duplicates(['country_code', 'year'])
gd = gd.merge(
    region_lookup,
    left_on=['code', 'year'],
    right_on=['country_code', 'year'],
    how='left',
    suffixes=('', '_from_fillgaps')
).drop(columns=['country_code'])

# Keep original region_code when present; otherwise use fillgaps-derived region.
if 'region_code' not in gd.columns and 'region_code_from_fillgaps' in gd.columns:
    gd['region_code'] = gd['region_code_from_fillgaps']
elif 'region_code' in gd.columns and 'region_code_from_fillgaps' in gd.columns:
    gd['region_code'] = gd['region_code'].fillna(gd['region_code_from_fillgaps'])

gd_final = gd[['year', 'code', 'region_code', 'obs', 'welf', 'pop', 'pipvintage']].copy()
gd_final.head(10)

,year,code,region_code,obs,welf,pop,pipvintage
0,1990,ABW,LCN,1,0.479785,0.000063,20260922_2021_01_02_PROD
1,1990,ABW,LCN,2,0.846876,0.000063,20260922_2021_01_02_PROD
2,1990,ABW,LCN,3,1.094172,0.000063,20260922_2021_01_02_PROD
3,1990,ABW,LCN,4,1.294011,0.000063,20260922_2021_01_02_PROD
4,1990,ABW,LCN,5,1.466517,0.000063,20260922_2021_01_02_PROD
5,1990,ABW,LCN,6,1.620634,0.000063,20260922_2021_01_02_PROD
6,1990,ABW,LCN,7,1.761288,0.000063,20260922_2021_01_02_PROD
7,1990,ABW,LCN,8,1.891536,0.000063,20260922_2021_01_02_PROD
8,1990,ABW,LCN,9,2.013431,0.000063,20260922_2021_01_02_PROD
9,1990,ABW,LCN,10,2.128432,0.000063,20260922_2021_01_02_PROD


## Chunk 7 - Integrity checks (balanced bins and missing welfare)
This chunk verifies if each country-year has 1000 bins and whether welfare has missing values.

In [9]:
qa = gd_final.groupby(['code', 'year'], as_index=False).agg(
    n_bins=('obs', 'nunique'),
    missing_welf=('welf', lambda s: int(s.isna().sum())),
    missing_pop=('pop', lambda s: int(s.isna().sum()))
)

summary = {
    'country_years': int(len(qa)),
    'country_years_not_1000_bins': int((qa['n_bins'] != 1000).sum()),
    'country_years_with_missing_welf': int((qa['missing_welf'] > 0).sum()),
    'country_years_with_missing_pop': int((qa['missing_pop'] > 0).sum())
}
summary

{'country_years': 8066,
 'country_years_not_1000_bins': 0,
 'country_years_with_missing_welf': 0,
 'country_years_with_missing_pop': 0}

## Chunk 7b - Malawi 1997 mean from global 1000 bins
This chunk computes Malawi's 1997 average welfare directly from `gd_final` and compares it against `fillgaps` for the same country-year/reporting level.

In [12]:
mwi_1997_bins = gd_final[(gd_final['code'] == 'MWI') & (gd_final['year'] == 1997)].copy()
mwi_1997_mean_bins = float(np.average(mwi_1997_bins['welf'], weights=mwi_1997_bins['pop']))

mwi_1997_fillgaps = fillgaps[
    (fillgaps['country_code'] == 'MWI')
    & (fillgaps['reporting_level'] == 'national')
    & (fillgaps['year'] == 1997)
][['country_code', 'year', 'reporting_level', 'mean']].copy()

mwi_1997_fillgaps_mean = float(mwi_1997_fillgaps['mean'].iloc[0]) if not mwi_1997_fillgaps.empty else np.nan

pd.DataFrame([
    {
        'code': 'MWI',
        'year': 1997,
        'mean_from_globaldist_bins': mwi_1997_mean_bins,
        'mean_from_fillgaps': mwi_1997_fillgaps_mean,
        'diff_bins_minus_fillgaps': mwi_1997_mean_bins - mwi_1997_fillgaps_mean,
        'n_bins': int(mwi_1997_bins['obs'].nunique()),
    }
])

,code,year,mean_from_globaldist_bins,mean_from_fillgaps,diff_bins_minus_fillgaps,n_bins
0,MWI,1997,4.865801,5.724018,-0.858217,1000


## Chunk 7c - Direct 2017 vs 2021 comparison for Malawi 1997
This chunk compares Malawi 1997 mean welfare computed from GlobalDist files built under PPP 2017 vs PPP 2021 (when both vintages are present in the workspace).

In [ ]:
from pathlib import Path

gd_files = sorted(ROOT.rglob('**/GlobalDist1000bins*.dta'))
ppp_files = {
    '2017': [p for p in gd_files if '_2017_' in p.name],
    '2021': [p for p in gd_files if '_2021_' in p.name],
}

rows = []
for ppp_year, files in ppp_files.items():
    if not files:
        rows.append({
            'ppp_year': ppp_year,
            'file_used': None,
            'status': 'missing file',
            'mwi_1997_mean_from_bins': np.nan,
        })
        continue

    # Use the latest file by name order within each PPP year.
    f = files[-1]
    d = pd.read_stata(f, columns=['code', 'year', 'welf', 'pop'], convert_categoricals=False)
    d['year'] = d['year'].astype(int)
    m = d[(d['code'] == 'MWI') & (d['year'] == 1997)].copy()

    if m.empty:
        rows.append({
            'ppp_year': ppp_year,
            'file_used': str(f),
            'status': 'MWI 1997 not found',
            'mwi_1997_mean_from_bins': np.nan,
        })
        continue

    rows.append({
        'ppp_year': ppp_year,
        'file_used': str(f),
        'status': 'ok',
        'mwi_1997_mean_from_bins': float(np.average(m['welf'], weights=m['pop'])),
    })

comparison_ppp = pd.DataFrame(rows).sort_values('ppp_year')
if comparison_ppp['status'].eq('ok').sum() == 2:
    v2017 = comparison_ppp.loc[comparison_ppp['ppp_year'] == '2017', 'mwi_1997_mean_from_bins'].iloc[0]
    v2021 = comparison_ppp.loc[comparison_ppp['ppp_year'] == '2021', 'mwi_1997_mean_from_bins'].iloc[0]
    comparison_ppp['diff_2021_minus_2017'] = np.where(
        comparison_ppp['ppp_year'] == '2021',
        v2021 - v2017,
        np.nan,
    )

comparison_ppp

## Chunk 8 - Optional branch if intermediate files become available
If `FullDistributions.dta` and `CollapsedDistributions.dta` are later added, this chunk will show the exact bridge to full Stata-style replication.

In [11]:
if fulldist_path is None or collapsed_path is None:
    print('Intermediate files are not available yet.')
    print('To do full 1:1 replication, add: FullDistributions.dta and CollapsedDistributions.dta (PPP 2021 run).')
else:
    fulldist = pd.read_stata(fulldist_path, convert_categoricals=False)
    collapsed = pd.read_stata(collapsed_path, convert_categoricals=False)
    print('FullDistributions shape:', fulldist.shape)
    print('CollapsedDistributions shape:', collapsed.shape)

Intermediate files are not available yet.
To do full 1:1 replication, add: FullDistributions.dta and CollapsedDistributions.dta (PPP 2021 run).


## Chunk 9 - Executive conclusion
- PPP 2021 vintage data is available for `fillgaps` and `GlobalDist1000bins`.
- The final global bins frame and QA checks are replicable with current files.
- Full Stata-equivalent intermediate transformation (from queried poverty lines to collapsed distributions) requires `FullDistributions.dta` and `CollapsedDistributions.dta`.
- Once those are provided in PPP 2021, this notebook can be extended to a full end-to-end replication.